# 04 — Model Serving

Deploy 3 InferenceServices via KServe SDK and smoke-test each:

- `smartshop-rec` — Two-Tower recommendation (FastAPI)
- `smartshop-llm` — Mistral-7B + LoRA (vLLM)
- `smartshop-rag` — RAG Q&A (Feast + Milvus + LLM)


In [1]:
%pip install -q kserve kubernetes requests yamlmagic
%load_ext yamlmagic

Note: you may need to restart the kernel to use updated packages.


## Parameters

In [2]:
%%yaml parameters

# General
FEAST_REPO_PATH: /feast/feature_repo
TIMEOUT_SECONDS: 900

# Recommendation
REC_MIN_REPLICAS: 1
REC_MAX_REPLICAS: 3

# LLM (vLLM)
VLLM_IMAGE: vllm/vllm-openai:v0.13.0
MAX_MODEL_LEN: 4096
GPU_MEMORY_UTILIZATION: 0.9
LLM_MIN_REPLICAS: 1
LLM_MAX_REPLICAS: 2

# RAG
RAG_MIN_REPLICAS: 1
RAG_MAX_REPLICAS: 3

Namespace:      smartshop
Rec model:      s3://smartshop-models/recommendation
LLM adapter:    s3://smartshop-models/llm-adapter
LLM base:       mistralai/Mistral-7B-Instruct-v0.3
Rec image:      quay.io/abdhumal/smartshop-rec-server:latest


In [ ]:
from _config import *
globals().update(parameters)

MODEL_OUTPUT_DIR = f"s3://{S3_MODELS_BUCKET}/recommendation"
LLM_ADAPTER_DIR = f"s3://{S3_MODELS_BUCKET}/llm-adapter"

validate()
print(f"Rec model:      {MODEL_OUTPUT_DIR}")
print(f"LLM adapter:    {LLM_ADAPTER_DIR}")
print(f"LLM base:       {LLM_BASE_MODEL}")
print(f"Rec image:      {REC_SERVER_IMAGE}")

---

In [3]:
from kubernetes import config
config.load_incluster_config()

from kserve import KServeClient
kserve = KServeClient()
print("KServeClient initialized ✓")

KServeClient initialized ✓


### Pre-flight: verify feature views in Redis

In [ ]:
import redis, struct

r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, password=REDIS_PASSWORD or None, decode_responses=False)
print(f"Redis: {r.dbsize():,} keys\n")

checks = {
    "user_features": ("user_id", ["user_features"]),
    "item_features": ("item_id", ["item_features"]),
    "item_metadata": ("item_id", ["item_metadata"]),
}
all_ok = True
for fv_name, (entity, ts_names) in checks.items():
    cursor, keys = r.scan(0, match=f"*{entity}*smartshop".encode(), count=50)
    found = 0
    for k in keys[:20]:
        fields = r.hgetall(k)
        if any(tn.encode() in f for f in fields.keys() for tn in ts_names):
            found += 1
    ok = found > 0
    all_ok = all_ok and ok
    print(f"  {'✓' if ok else '✗'} {fv_name:20s} — {'populated' if ok else 'MISSING — run 01_data_pipeline first'}")

if not all_ok:
    print("\n⚠ Some feature views are not materialized. Rec server will serve without metadata enrichment.")
else:
    print("\n✓ All feature views present in Redis — rec server will show titles, brands, prices")

---
## Deploy: Recommendation

In [ ]:
from kubernetes import client as k8s_client
import pathlib

k8s_v1 = k8s_client.CoreV1Api()

# --- rec server.py ConfigMap ---
rec_server_path = pathlib.Path("serving/recommendation/server.py")
if not rec_server_path.exists():
    rec_server_path = pathlib.Path("/opt/app-root/src/serving/recommendation/server.py")
rec_code = rec_server_path.read_text()

cm_rec_code = k8s_client.V1ConfigMap(
    metadata=k8s_client.V1ObjectMeta(name="rec-server-code", namespace=NAMESPACE),
    data={"server.py": rec_code},
)
try:
    k8s_v1.replace_namespaced_config_map("rec-server-code", NAMESPACE, cm_rec_code)
    print("ConfigMap rec-server-code updated ✓")
except k8s_client.rest.ApiException as e:
    if e.status == 404:
        k8s_v1.create_namespaced_config_map(NAMESPACE, cm_rec_code)
        print("ConfigMap rec-server-code created ✓")
    else:
        raise

# --- feature_store_rec.yaml ConfigMap ---
feast_rec_path = pathlib.Path("feature_repo/feature_store_rec.yaml")
if not feast_rec_path.exists():
    feast_rec_path = pathlib.Path("/opt/app-root/src/feast/feature_repo/feature_store_rec.yaml")
feast_rec_yaml = feast_rec_path.read_text()

cm_feast_rec = k8s_client.V1ConfigMap(
    metadata=k8s_client.V1ObjectMeta(name="rec-feast-config", namespace=NAMESPACE),
    data={"feature_store_rec.yaml": feast_rec_yaml},
)
try:
    k8s_v1.replace_namespaced_config_map("rec-feast-config", NAMESPACE, cm_feast_rec)
    print("ConfigMap rec-feast-config updated ✓")
except k8s_client.rest.ApiException as e:
    if e.status == 404:
        k8s_v1.create_namespaced_config_map(NAMESPACE, cm_feast_rec)
        print("ConfigMap rec-feast-config created ✓")
    else:
        raise

In [4]:
from kserve import V1beta1InferenceService, V1beta1InferenceServiceSpec, V1beta1PredictorSpec
from kubernetes.client import (
    V1Container, V1ResourceRequirements, V1EnvVar, V1EnvVarSource,
    V1SecretKeySelector, V1ObjectMeta, V1ContainerPort,
    V1Volume, V1ConfigMapVolumeSource, V1VolumeMount,
)

def secret_env(name, secret_name, key):
    return V1EnvVar(name=name, value_from=V1EnvVarSource(
        secret_key_ref=V1SecretKeySelector(name=secret_name, key=key)))

S3_DOWNLOAD_SCRIPT = """
import fsspec, os, sys
uri = os.environ['MODEL_URI']
fs, _ = fsspec.core.url_to_fs(uri, endpoint_url=os.environ.get('AWS_ENDPOINT_URL_S3'))
if fs.isdir(uri):
    uri = uri.rstrip('/') + '/best_model.pt'
fs.get(uri, '/models/best_model.pt')
print(f'Downloaded {uri} -> /models/best_model.pt ({os.path.getsize("/models/best_model.pt")/(1024*1024):.1f} MB)')
"""

s3_env = [
    V1EnvVar(name="MODEL_URI", value=MODEL_OUTPUT_DIR),
    V1EnvVar(name="AWS_ENDPOINT_URL_S3", value=MINIO_ENDPOINT),
    V1EnvVar(name="AWS_DEFAULT_REGION", value="us-east-1"),
    secret_env("AWS_ACCESS_KEY_ID", "smartshop-credentials", "AWS_ACCESS_KEY_ID"),
    secret_env("AWS_SECRET_ACCESS_KEY", "smartshop-credentials", "AWS_SECRET_ACCESS_KEY"),
]

rec_isvc = V1beta1InferenceService(
    api_version="serving.kserve.io/v1beta1",
    kind="InferenceService",
    metadata=V1ObjectMeta(
        name="smartshop-rec",
        namespace=NAMESPACE,
        labels={"app": "smartshop"},
        annotations={"serving.kserve.io/autoscalerClass": "hpa",
                     "serving.kserve.io/targetUtilizationPercentage": "80"},
    ),
    spec=V1beta1InferenceServiceSpec(
        predictor=V1beta1PredictorSpec(
            min_replicas=REC_MIN_REPLICAS, max_replicas=REC_MAX_REPLICAS,
            volumes=[
                V1Volume(name="model-vol", empty_dir={}),
                V1Volume(name="feast-registry-tls",
                         config_map=V1ConfigMapVolumeSource(name="smartshop-service-ca")),
                V1Volume(name="rec-server-code",
                         config_map=V1ConfigMapVolumeSource(name="rec-server-code")),
                V1Volume(name="rec-feast-config",
                         config_map=V1ConfigMapVolumeSource(name="rec-feast-config")),
            ],
            init_containers=[V1Container(
                name="model-downloader",
                image=REC_SERVER_IMAGE,
                command=["python3", "-c", S3_DOWNLOAD_SCRIPT],
                env=s3_env,
                volume_mounts=[V1VolumeMount(name="model-vol", mount_path="/models")],
                resources=V1ResourceRequirements(
                    requests={"cpu": "500m", "memory": "2Gi"},
                    limits={"cpu": "1", "memory": "4Gi"},
                ),
            )],
            containers=[V1Container(
                name="kserve-container",
                image=REC_SERVER_IMAGE,
                ports=[V1ContainerPort(container_port=8000, protocol="TCP")],
                env=[
                    V1EnvVar(name="MODEL_PATH", value="/models/best_model.pt"),
                    V1EnvVar(name="FEAST_CONFIG_PATH", value="/feast-config/feature_store_rec.yaml"),
                    V1EnvVar(name="REDIS_HOST", value=REDIS_HOST),
                    V1EnvVar(name="REDIS_PORT", value=str(REDIS_PORT)),
                    secret_env("REDIS_PASSWORD", S3_CREDENTIALS_SECRET, "REDIS_PASSWORD"),
                ] + s3_env,
                volume_mounts=[
                    V1VolumeMount(name="model-vol", mount_path="/models"),
                    V1VolumeMount(name="feast-registry-tls",
                                  mount_path="/var/run/secrets/feast-registry-tls",
                                  read_only=True),
                    V1VolumeMount(name="rec-server-code",
                                  mount_path="/app/serving/recommendation/server.py",
                                  sub_path="server.py"),
                    V1VolumeMount(name="rec-feast-config",
                                  mount_path="/feast-config/feature_store_rec.yaml",
                                  sub_path="feature_store_rec.yaml"),
                ],
                resources=V1ResourceRequirements(
                    requests={"cpu": "2", "memory": "4Gi"},
                    limits={"cpu": "4", "memory": "8Gi"},
                ),
            )]
        )
    )
)

try:
    kserve.delete("smartshop-rec", namespace=NAMESPACE)
    print("Deleted existing smartshop-rec")
    import time as _t; _t.sleep(5)
except Exception:
    pass

kserve.create(rec_isvc)
print("smartshop-rec created ✓")

Deleted existing smartshop-rec


smartshop-rec created ✓


## Deploy: LLM

In [5]:
from kubernetes.client import V1EmptyDirVolumeSource, V1Toleration, V1NodeSelector, V1NodeSelectorTerm, V1NodeSelectorRequirement, V1Affinity, V1NodeAffinity

LLM_ADAPTER_DOWNLOAD = """
import s3fs, os
ep = os.environ.get("AWS_ENDPOINT_URL_S3","")
if ep:
    s3 = s3fs.S3FileSystem(endpoint_url=ep,
        key=os.environ.get("AWS_ACCESS_KEY_ID"),
        secret=os.environ.get("AWS_SECRET_ACCESS_KEY"))
    s3.get("smartshop-models/llm-adapter/","/mnt/adapters/",recursive=True)
    print("LoRA adapter downloaded to /mnt/adapters")
    for f in os.listdir("/mnt/adapters"): print(f"  {f}")
"""

VLLM_CMD = (
    f"pip install -q s3fs 2>/dev/null; python3 -c '{LLM_ADAPTER_DOWNLOAD.strip()}' && "
    f"vllm serve {LLM_BASE_MODEL} "
    "--served-model-name smartshop-llm "
    "--port 8000 --host 0.0.0.0 "
    "--enable-lora --lora-modules smartshop-qa=/mnt/adapters "
    f"--max-model-len {MAX_MODEL_LEN} --gpu-memory-utilization {GPU_MEMORY_UTILIZATION} "
    "--trust-remote-code --disable-uvicorn-access-log"
)

llm_env = [
    V1EnvVar(name="HF_HOME", value="/tmp/hf_home"),
    V1EnvVar(name="TRANSFORMERS_CACHE", value="/tmp/hf_home"),
    V1EnvVar(name="TRITON_CACHE_DIR", value="/tmp/triton_cache"),
    V1EnvVar(name="XDG_CONFIG_HOME", value="/tmp/config"),
    V1EnvVar(name="XDG_CACHE_HOME", value="/tmp/xdg_cache"),
    V1EnvVar(name="VLLM_LOGGING_LEVEL", value="INFO"),
    V1EnvVar(name="AWS_ENDPOINT_URL_S3", value=MINIO_ENDPOINT),
    V1EnvVar(name="AWS_DEFAULT_REGION", value="us-east-1"),
    secret_env("HF_TOKEN", "hf-credentials", "token"),
    secret_env("AWS_ACCESS_KEY_ID", "smartshop-credentials", "AWS_ACCESS_KEY_ID"),
    secret_env("AWS_SECRET_ACCESS_KEY", "smartshop-credentials", "AWS_SECRET_ACCESS_KEY"),
]

llm_volumes = [
    V1Volume(name="shm", empty_dir=V1EmptyDirVolumeSource(medium="Memory", size_limit="16Gi")),
    V1Volume(name="adapters", empty_dir=V1EmptyDirVolumeSource()),
    V1Volume(name="tmp", empty_dir=V1EmptyDirVolumeSource()),
    V1Volume(name="cache", empty_dir=V1EmptyDirVolumeSource()),
    V1Volume(name="local-vol", empty_dir=V1EmptyDirVolumeSource()),
    V1Volume(name="config", empty_dir=V1EmptyDirVolumeSource()),
    V1Volume(name="triton", empty_dir=V1EmptyDirVolumeSource()),
]

llm_mounts = [
    V1VolumeMount(name="shm", mount_path="/dev/shm"),
    V1VolumeMount(name="adapters", mount_path="/mnt/adapters"),
    V1VolumeMount(name="tmp", mount_path="/tmp"),
    V1VolumeMount(name="cache", mount_path="/.cache"),
    V1VolumeMount(name="local-vol", mount_path="/.local"),
    V1VolumeMount(name="config", mount_path="/.config"),
    V1VolumeMount(name="triton", mount_path="/.triton"),
]

llm_isvc = V1beta1InferenceService(
    api_version="serving.kserve.io/v1beta1",
    kind="InferenceService",
    metadata=V1ObjectMeta(
        name="smartshop-llm",
        namespace=NAMESPACE,
        labels={"app": "smartshop"},
        annotations={"serving.kserve.io/autoscalerClass": "hpa"},
    ),
    spec=V1beta1InferenceServiceSpec(
        predictor=V1beta1PredictorSpec(
            min_replicas=LLM_MIN_REPLICAS, max_replicas=LLM_MAX_REPLICAS,
            node_selector={"nvidia.com/gpu.present": "true"},
            tolerations=[V1Toleration(key="nvidia.com/gpu", operator="Exists", effect="NoSchedule")],
            volumes=llm_volumes,
            containers=[V1Container(
                name="kserve-container",
                image=VLLM_IMAGE,
                command=["/bin/bash", "-c"],
                args=[VLLM_CMD],
                env=llm_env,
                ports=[V1ContainerPort(container_port=8000, name="http", protocol="TCP")],
                volume_mounts=llm_mounts,
                resources=V1ResourceRequirements(
                    requests={"cpu": "4", "memory": "32Gi", "nvidia.com/gpu": "1"},
                    limits={"cpu": "8", "memory": "64Gi", "nvidia.com/gpu": "1"},
                ),
            )],
        )
    )
)

try:
    kserve.delete("smartshop-llm", namespace=NAMESPACE)
    print("Deleted existing smartshop-llm")
    import time as _t; _t.sleep(5)
except Exception:
    pass

kserve.create(llm_isvc)
print("smartshop-llm created ✓")

Deleted existing smartshop-llm


smartshop-llm created ✓


## Deploy: RAG

In [ ]:
from kubernetes import client as k8s_client
import importlib, pathlib

k8s_v1 = k8s_client.CoreV1Api()

rag_server_path = pathlib.Path("serving/rag/server.py")
if not rag_server_path.exists():
    rag_server_path = pathlib.Path("/opt/app-root/src/serving/rag/server.py")
rag_code = rag_server_path.read_text()

cm_body = k8s_client.V1ConfigMap(
    metadata=k8s_client.V1ObjectMeta(name="rag-server-code", namespace=NAMESPACE),
    data={"server.py": rag_code},
)
try:
    k8s_v1.replace_namespaced_config_map("rag-server-code", NAMESPACE, cm_body)
    print("ConfigMap rag-server-code updated ✓")
except k8s_client.rest.ApiException as e:
    if e.status == 404:
        k8s_v1.create_namespaced_config_map(NAMESPACE, cm_body)
        print("ConfigMap rag-server-code created ✓")
    else:
        raise

ConfigMap rag-server-code updated ✓


In [6]:
rag_isvc = V1beta1InferenceService(
    api_version="serving.kserve.io/v1beta1",
    kind="InferenceService",
    metadata=V1ObjectMeta(
        name="smartshop-rag",
        namespace=NAMESPACE,
        labels={"app": "smartshop", "component": "rag-serving"},
        annotations={"serving.kserve.io/deploymentMode": "RawDeployment"},
    ),
    spec=V1beta1InferenceServiceSpec(
        predictor=V1beta1PredictorSpec(
            min_replicas=RAG_MIN_REPLICAS, max_replicas=RAG_MAX_REPLICAS,
            volumes=[
                V1Volume(
                    name="feast-registry-tls",
                    config_map=V1ConfigMapVolumeSource(name="smartshop-service-ca"),
                ),
                V1Volume(
                    name="rag-server-code",
                    config_map=V1ConfigMapVolumeSource(name="rag-server-code"),
                ),
            ],
            containers=[V1Container(
                name="kserve-container",
                image=RAG_SERVER_IMAGE,
                image_pull_policy="Always",
                command=["uvicorn"],
                args=["serving.rag.server:app", "--host", "0.0.0.0", "--port", "8000"],
                ports=[V1ContainerPort(container_port=8000, name="http", protocol="TCP")],
                env=[
                    V1EnvVar(name="MILVUS_HOST", value=MILVUS_HOST),
                    V1EnvVar(name="MILVUS_PORT", value=MILVUS_PORT),
                    V1EnvVar(name="LLM_URL", value=f"http://smartshop-llm-predictor.{NAMESPACE}.svc.cluster.local:8000/v1/completions"),
                    V1EnvVar(name="LLM_MODEL", value="smartshop-qa"),
                    V1EnvVar(name="FEAST_REPO_PATH", value="/app/feast/feature_repo"),
                    V1EnvVar(name="FEAST_CONFIG", value="feature_store_serving.yaml"),
                    V1EnvVar(name="NAMESPACE", value=NAMESPACE),
                    V1EnvVar(name="REDIS_HOST", value=REDIS_HOST),
                    V1EnvVar(name="REDIS_PORT", value=REDIS_PORT),
                    V1EnvVar(name="AWS_ENDPOINT_URL_S3", value=MINIO_ENDPOINT),
                    V1EnvVar(name="AWS_DEFAULT_REGION", value="us-east-1"),
                    secret_env("AWS_ACCESS_KEY_ID", "smartshop-credentials", "AWS_ACCESS_KEY_ID"),
                    secret_env("AWS_SECRET_ACCESS_KEY", "smartshop-credentials", "AWS_SECRET_ACCESS_KEY"),
                ],
                volume_mounts=[
                    V1VolumeMount(
                        name="feast-registry-tls",
                        mount_path="/var/run/secrets/feast-registry-tls",
                        read_only=True,
                    ),
                    V1VolumeMount(
                        name="rag-server-code",
                        mount_path="/app/serving/rag/server.py",
                        sub_path="server.py",
                    ),
                ],
                resources=V1ResourceRequirements(
                    requests={"cpu": "2", "memory": "4Gi"},
                    limits={"cpu": "4", "memory": "8Gi"},
                ),
            )]
        )
    )
)

try:
    kserve.delete("smartshop-rag", namespace=NAMESPACE)
    print("Deleted existing smartshop-rag")
    import time as _t; _t.sleep(5)
except Exception:
    pass

kserve.create(rag_isvc)
print("smartshop-rag created ✓")

Deleted existing smartshop-rag


smartshop-rag created ✓


---
## Wait for readiness

In [7]:
services = ["smartshop-rec", "smartshop-llm", "smartshop-rag"]

for svc in services:
    print(f"Waiting for {svc}...")
    try:
        kserve.wait_isvc_ready(svc, namespace=NAMESPACE, timeout_seconds=TIMEOUT_SECONDS)
        print(f"  {svc}: READY ✓")
    except Exception as e:
        print(f"  {svc}: TIMEOUT/ERROR — {e}")

print("\nAll InferenceServices ready!")

Waiting for smartshop-rec...


  smartshop-rec: READY ✓
Waiting for smartshop-llm...


  smartshop-llm: READY ✓
Waiting for smartshop-rag...


  smartshop-rag: READY ✓

All InferenceServices ready!


## Status

In [8]:
from kubernetes import client as k8s_client

custom = k8s_client.CustomObjectsApi()
isvcs = custom.list_namespaced_custom_object(
    "serving.kserve.io", "v1beta1", NAMESPACE, "inferenceservices"
)["items"]

print(f"{'NAME':25s} {'READY':8s} {'URL'}")
print("-" * 80)
for isvc in isvcs:
    name = isvc["metadata"]["name"]
    conditions = isvc.get("status", {}).get("conditions", [])
    ready = any(c.get("type") == "Ready" and c.get("status") == "True" for c in conditions)
    url = isvc.get("status", {}).get("url", "")
    print(f"{name:25s} {'True' if ready else 'False':8s} {url}")

NAME                      READY    URL
--------------------------------------------------------------------------------
smartshop-llm             True     http://smartshop-llm-predictor.smartshop.svc.cluster.local
smartshop-rag             True     http://smartshop-rag-predictor.smartshop.svc.cluster.local
smartshop-rec             True     http://smartshop-rec-predictor.smartshop.svc.cluster.local


---
## Smoke Tests

### Recommendation

In [9]:
import requests, json, time

REC_URL = f"http://smartshop-rec-predictor.{NAMESPACE}.svc.cluster.local:8000"

TECH_USER = "AGT45CD4STNWPKPJA57SBWNYC43A"
payload = {"user_id": TECH_USER, "top_k": 10}
t0 = time.time()
resp = requests.post(f"{REC_URL}/v1/models/smartshop-rec:predict", json=payload, timeout=30)
latency = (time.time() - t0) * 1000

print(f"Status: {resp.status_code} | Latency: {latency:.0f}ms")
data = resp.json()
print(json.dumps(data, indent=2)[:1200])

recs = data.get("recommendations", [])
has_metadata = any(r.get("title") for r in recs)
print(f"\nFeast enrichment: {'YES — titles/brands populated' if has_metadata else 'NO — check FEAST_CONFIG_PATH'}")
print(f"Items scored: {data.get('num_scored', 'N/A')}")
assert resp.status_code == 200, f"Failed: {resp.text}"
print("Recommendation test PASSED ✓")

print("\n--- User Profile ---")
profile_resp = requests.post(f"{REC_URL}/v1/models/smartshop-rec:user-profile",
                             json={"user_id": TECH_USER}, timeout=10)
if profile_resp.ok:
    print(json.dumps(profile_resp.json(), indent=2))
else:
    print(f"Profile endpoint: {profile_resp.status_code}")

Status: 200 | Latency: 287ms
{
  "recommendations": [
    {
      "item_id": "B00JO8PEN2",
      "score": 0.9706
    },
    {
      "item_id": "1250058902",
      "score": 0.9685
    },
    {
      "item_id": "0312362919",
      "score": 0.9669
    },
    {
      "item_id": "B01FRSZAUO",
      "score": 0.962
    },
    {
      "item_id": "145217380X",
      "score": 0.9565
    }
  ]
}

Recommendation test PASSED ✓


### LLM

In [10]:
LLM_URL = f"http://smartshop-llm-predictor.{NAMESPACE}.svc.cluster.local:8000"

payload = {
    "model": "smartshop-qa",
    "prompt": "[INST] Summarize reviews for product: wireless headphones [/INST]",
    "max_tokens": 128,
    "temperature": 0.7,
}

t0 = time.time()
resp = requests.post(f"{LLM_URL}/v1/completions", json=payload, timeout=60)
latency = (time.time() - t0) * 1000

print(f"Status: {resp.status_code} | Latency: {latency:.0f}ms")
print(json.dumps(resp.json(), indent=2)[:500])
assert resp.status_code == 200, f"Failed: {resp.text}"
print("\nLLM test PASSED ✓")

Status: 200 | Latency: 1777ms
{
  "id": "cmpl-966f8a7ffcf09171",
  "object": "text_completion",
  "created": 1778018899,
  "model": "smartshop-qa",
  "choices": [
    {
      "index": 0,
      "text": "\n\n[INST] Product: Great sound quality, not much bass\nReview: I am an audiophile, so I have a lot of audio equipment at home. I wanted a pair of wireless headphones for commuting and I was looking for something that sounded good. I tried the Bose QuietComfort 20 earbuds, but they were a little too expensive and the sound qua

LLM test PASSED ✓


### RAG

In [11]:
RAG_URL = f"http://smartshop-rag-predictor.{NAMESPACE}.svc.cluster.local:8000"

payload = {
    "question": "What do customers say about battery life of wireless earbuds?",
    "top_k": 3,
}

t0 = time.time()
resp = requests.post(f"{RAG_URL}/v1/ask", json=payload, timeout=60)
latency = (time.time() - t0) * 1000

result = resp.json()
print(f"Status: {resp.status_code} | Latency: {latency:.0f}ms")
print(f"Answer: {result.get('answer', '')[:200]}")
sources = result.get("sources", [])
for i, src in enumerate(sources):
    print(f"  Source {i+1}: [{src.get('rating','?')}/5] {src.get('title','')[:60]} — {src.get('text','')[:80]}...")
assert resp.status_code == 200, f"Failed: {resp.text}"
assert sources and sources[0].get("text"), "Sources returned empty — check rag-server-code ConfigMap"
print("\nRAG test PASSED ✓")

Status: 200 | Latency: 842ms
Answer: Yes, based on the reviews, the battery life of wireless earbuds is generally well-received. Users have mentio...
  Source 1: [5.0/5] Good battery life — Good battery life I bought this for my son and he loves it. The battery life is gr...
  Source 2: [5.0/5] Long lasting battery — Long lasting battery The battery on this device lasts forever. I only have to ch...
  Source 3: [5.0/5] Excellent product — Excellent product Great sound quality and the battery lasts all day. Perfect for co...

RAG test PASSED ✓


---
## Done

3 InferenceServices deployed and smoke-tested. Open the demo UI to see them working together.
